In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

months = [
    "202502", "202503", "202504", "202505",
    "202506", "202507", "202508", "202509",
    "202510", "202511", "202512", "202601",
    "202602", "202603", "202604", "202605"
]

dfs = [
    pd.read_csv(f"/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold{m}.csv")
    for m in months
]

/tmp/ipykernel_11991/3680537881.py:11: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(f"/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold{m}.csv")
/tmp/ipykernel_11991/3680537881.py:11: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(f"/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold{m}.csv")


In [4]:
df_original = pd.concat(dfs, ignore_index=True)
df = df_original.copy()

In [5]:
df_sf = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

In [6]:
df_sf["CloseDate"] = pd.to_datetime(df_sf["CloseDate"], errors="coerce")

#Features

In [7]:
location_features = [
    "Latitude",
    "Longitude",
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]

property_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "GarageSpaces",
    "ParkingTotal",
]

lot_financial_features = [
    "AssociationFee",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
]

#Target

In [8]:
target = "ClosePrice"
date_col = "CloseDate"

all_features = location_features + property_features + lot_financial_features
required_columns = all_features + [target, date_col]

In [9]:
missing_columns = [col for col in required_columns if col not in df_sf.columns]
df_model = df_sf[required_columns].copy()

print("Input feature count:", len(all_features))
print("Selected dataset shape:", df_model.shape)

Input feature count: 21
Selected dataset shape: (173338, 23)


In [10]:
df_model = df_model.dropna(subset=[date_col]).copy()

train_df = df_model[
    (df_model[date_col] >= "2025-02-01") &
    (df_model[date_col] < "2026-02-01")
].copy()

validation_df = df_model[
    (df_model[date_col] >= "2026-02-01") &
    (df_model[date_col] < "2026-05-01")
].copy()

test_df = df_model[
    (df_model[date_col] >= "2026-05-01") &
    (df_model[date_col] < "2026-06-01")
].copy()

print("Training period:", train_df[date_col].min(), "to", train_df[date_col].max())
print("Validation period:", validation_df[date_col].min(), "to", validation_df[date_col].max())
print("Testing period:", test_df[date_col].min(), "to", test_df[date_col].max())

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)


Training period: 2025-02-01 00:00:00 to 2026-01-31 00:00:00
Validation period: 2026-02-01 00:00:00 to 2026-04-30 00:00:00
Testing period: 2026-05-01 00:00:00 to 2026-05-31 00:00:00
Train shape: (129556, 23)
Validation shape: (31758, 23)
Test shape: (12024, 23)


# **Handle missing values**

In [11]:
train_missing = pd.DataFrame({
    "Missing Count": train_df[all_features].isna().sum(),
    "Missing %": (train_df[all_features].isna().mean() * 100).round(2),
    "Dtype": train_df[all_features].dtypes.astype(str),
}).sort_values("Missing %", ascending=False)

train_missing

,Missing Count,Missing %,Dtype
AssociationFee,37723,29.12,float64
HighSchoolDistrict,34762,26.83,object
MLSAreaMajor,18861,14.56,object
AttachedGarageYN,15635,12.07,object
Stories,13645,10.53,float64
ViewYN,11800,9.11,object
PoolPrivateYN,9956,7.68,object
GarageSpaces,5185,4.00,float64
LotSizeAcres,2227,1.72,float64
LotSizeSquareFeet,2227,1.72,float64


In [12]:
numeric_features = [
    "Latitude",
    "Longitude",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "GarageSpaces",
    "ParkingTotal",
    "AssociationFee",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
]

high_cardinality_categorical = [
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]

low_cardinality_categorical = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
]

In [13]:
for frame in [train_df, test_df]:
    frame["ParkingTotal"] = frame["ParkingTotal"].where(
        frame["ParkingTotal"] >= 0,
        np.nan
    )

In [14]:
train_medians = train_df[numeric_features].median()

for col in numeric_features:
    for frame in [train_df, validation_df, test_df]:
        frame[f"{col}_missing"] = frame[col].isna().astype(int)
        frame[col] = frame[col].fillna(train_medians[col])

In [15]:
for col in high_cardinality_categorical + low_cardinality_categorical:
    train_df[col] = train_df[col].fillna("Missing").astype(str)
    validation_df[col] = validation_df[col].fillna("Missing").astype(str)
    test_df[col] = test_df[col].fillna("Missing").astype(str)

In [16]:
for frame in [train_df, validation_df, test_df]:
    frame["PostalCode"] = frame["PostalCode"].astype(str)

print("Remaining training-set missing values:")
print(train_df[all_features].isna().sum().sort_values(ascending=False).head())

print("\nRemaining validation-set missing values:")
print(validation_df[all_features].isna().sum().sort_values(ascending=False).head())

print("\nRemaining test-set missing values:")
print(test_df[all_features].isna().sum().sort_values(ascending=False).head())

Remaining training-set missing values:
Latitude          0
Longitude         0
City              0
PostalCode        0
CountyOrParish    0
dtype: int64

Remaining validation-set missing values:
Latitude          0
Longitude         0
City              0
PostalCode        0
CountyOrParish    0
dtype: int64

Remaining test-set missing values:
Latitude          0
Longitude         0
City              0
PostalCode        0
CountyOrParish    0
dtype: int64


In [17]:
y_train = train_df[target].copy()
y_validation = validation_df[target].copy()
y_test = test_df[target].copy()

target_encoder = TargetEncoder(
    target_type="continuous",
    smooth="auto",
    random_state=42
)

train_high_encoded = target_encoder.fit_transform(
    train_df[high_cardinality_categorical],
    y_train
)
validation_high_encoded = target_encoder.transform(
    validation_df[high_cardinality_categorical]
)
test_high_encoded = target_encoder.transform(
    test_df[high_cardinality_categorical]
)

encoded_names = [
    f"{col}_encoded" for col in high_cardinality_categorical
]

train_high_encoded = pd.DataFrame(
    train_high_encoded,
    columns=encoded_names,
    index=train_df.index
)
validation_high_encoded = pd.DataFrame(
    validation_high_encoded,
    columns=encoded_names,
    index=validation_df.index
)
test_high_encoded = pd.DataFrame(
    test_high_encoded,
    columns=encoded_names,
    index=test_df.index
)

In [18]:
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

train_low_encoded = onehot_encoder.fit_transform(
    train_df[low_cardinality_categorical]
)
validation_low_encoded = onehot_encoder.transform(
    validation_df[low_cardinality_categorical]
)
test_low_encoded = onehot_encoder.transform(
    test_df[low_cardinality_categorical]
)

onehot_names = onehot_encoder.get_feature_names_out(
    low_cardinality_categorical
)

train_low_encoded = pd.DataFrame(
    train_low_encoded,
    columns=onehot_names,
    index=train_df.index
)
validation_low_encoded = pd.DataFrame(
    validation_low_encoded,
    columns=onehot_names,
    index=validation_df.index
)
test_low_encoded = pd.DataFrame(
    test_low_encoded,
    columns=onehot_names,
    index=test_df.index
)

print("Target-encoded columns:", encoded_names)
print("One-hot encoded columns:", onehot_names.tolist())

Target-encoded columns: ['City_encoded', 'PostalCode_encoded', 'CountyOrParish_encoded', 'MLSAreaMajor_encoded', 'HighSchoolDistrict_encoded']
One-hot encoded columns: ['ViewYN_Missing', 'ViewYN_True', 'PoolPrivateYN_Missing', 'PoolPrivateYN_True', 'AttachedGarageYN_Missing', 'AttachedGarageYN_True']


In [19]:
raw_numeric_features = numeric_features

missing_flag_features = [
    f"{col}_missing" for col in numeric_features
]

X_train = pd.concat(
    [
        train_df[raw_numeric_features + missing_flag_features],
        train_high_encoded,
        train_low_encoded,
    ],
    axis=1
)

X_validation = pd.concat(
    [
        validation_df[raw_numeric_features + missing_flag_features],
        validation_high_encoded,
        validation_low_encoded,
    ],
    axis=1
)

X_test = pd.concat(
    [
        test_df[raw_numeric_features + missing_flag_features],
        test_high_encoded,
        test_low_encoded,
    ],
    axis=1
)

X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Final feature count:", X_train.shape[1])
print("Non-numeric columns remaining:",
      X_train.select_dtypes(exclude="number").columns.tolist())
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

Final feature count: 37
Non-numeric columns remaining: []
X_train shape: (129556, 37)
X_validation shape: (31758, 37)
X_test shape: (12024, 37)


In [20]:
train_export = X_train.copy()
train_export[target] = y_train.values
train_export[date_col] = train_df[date_col].values
train_export["split"] = "train"

validation_export = X_validation.copy()
validation_export[target] = y_validation.values
validation_export[date_col] = validation_df[date_col].values
validation_export["split"] = "validation"

test_export = X_test.copy()
test_export[target] = y_test.values
test_export[date_col] = test_df[date_col].values
test_export["split"] = "test"

train_export = train_export[train_export[target] > 0].copy()
validation_export = validation_export[validation_export[target] > 0].copy()
test_export = test_export[test_export[target] > 0].copy()

cleaned_data = pd.concat(
    [train_export, validation_export, test_export],
    ignore_index=True
)

In [21]:
train_df.to_csv("train_preprocessed.csv", index=False)
validation_df.to_csv("validation_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)

train_export.to_csv("cleaned_train.csv", index=False)
validation_export.to_csv("cleaned_validation.csv", index=False)
test_export.to_csv("cleaned_test.csv", index=False)
cleaned_data.to_csv("cleaned_data.csv", index=False)

print("train_preprocessed.csv:", train_df.shape)
print("validation_preprocessed.csv:", validation_df.shape)
print("test_preprocessed.csv:", test_df.shape)

print("cleaned_train.csv:", train_export.shape)
print("cleaned_validation.csv:", validation_export.shape)
print("cleaned_test.csv:", test_export.shape)
print("cleaned_data.csv:", cleaned_data.shape)

train_preprocessed.csv: (129556, 36)
validation_preprocessed.csv: (31758, 36)
test_preprocessed.csv: (12024, 36)
cleaned_train.csv: (129555, 40)
cleaned_validation.csv: (31758, 40)
cleaned_test.csv: (12024, 40)
cleaned_data.csv: (173337, 40)
